[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc5_abtest/cours/seance1_cours.ipynb)

# Séance 5.1 — Causalité et A/B testing — mesurer ce qu'une campagne fait vraiment

**Cours** · durée : 6h (2h de cours, 2h d'étude de cas, 2h de correction)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- distinguer une question prédictive d'une question causale
- nommer le contrefactuel, l'ATE et l'ATT, et dire pourquoi on ne les observe jamais
- décomposer une comparaison de moyennes en effet causal + biais de sélection
- vérifier qu'un tirage au sort a fonctionné avec un tableau d'équilibre
- chiffrer l'effet d'un A/B test, son incertitude, et le traduire en décision

## Une slide, un lundi matin

Vous êtes l'équipe data d'un site de vente en ligne. La directrice marketing
vous transfère la slide d'un stagiaire, très enthousiaste :

> 💬 *« Résultat spectaculaire : les clients qui ont reçu l'email hommes **et
> visité le site** dépensent en moyenne **7,78 €**, contre **0,65 €** pour le
> groupe sans email. Nos emails multiplient les dépenses par 12 ! Il faut
> généraliser immédiatement. »*

Les deux chiffres sont **exacts**. Nous les recalculerons ensemble, ils
tombent juste. La conclusion, elle, est fausse — et l'entreprise s'apprête à
engager un budget dessus.

Ce cours sert à savoir dire **pourquoi**, et à le dire avec des chiffres
plutôt qu'avec une intuition.

## Prédire n'est pas causer

Le bloc 4 vous a appris à **prédire**. C'est ce que le machine learning fait
très bien : reconnaître un visage, estimer un panier, repérer un client qui
va résilier.

Mais une entreprise ne prend jamais une décision purement prédictive. Elle
demande : *si nous faisions X, que se passerait-il ?* C'est une question
**causale**, et un excellent modèle prédictif peut y répondre n'importe quoi.

L'exemple classique est l'hôtellerie. Dans les données, **prix bas ↔ faibles
ventes** : les hôtels bradent hors saison. Un modèle prédictif apprend cette
association et suggère, en toute logique, qu'**augmenter les prix ferait
vendre plus de chambres**. Le modèle n'a pas tort sur les données. Il répond
simplement à une autre question que celle qu'on lui pose.

| La question | Le type | L'outil |
|---|---|---|
| Quels clients vont acheter ? | prédictive | bloc 4 |
| Combien va dépenser ce client ? | prédictive | bloc 4 |
| **Envoyer un email augmente-t-il les achats ?** | **causale** | ce bloc |
| **Que dépenseraient-ils si on ne leur écrivait pas ?** | **causale** | ce bloc |

Le test qui tranche : **la question contient-elle une intervention, un « si on
faisait X » ?** Si oui, aucune corrélation ne suffira.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/data/"

## Des tablettes à l'école

Un ministère veut savoir si distribuer des tablettes améliore les résultats
scolaires. On observe que **les écoles qui distribuent des tablettes ont de
meilleurs scores**. Faut-il généraliser ?

Vous sentez le piège : ces écoles sont probablement plus riches. Elles
auraient de meilleurs scores **de toute façon**.

Simulons ce monde. L'intérêt d'une simulation, c'est qu'on y connaît la
vérité — ici, nous **décidons** que la tablette fait perdre 50 points.

In [ ]:
np.random.seed(123)
n = 1000
frais = np.random.normal(1000, 300, n).round()

# Les ecoles cheres distribuent des tablettes bien plus souvent : 80 % contre 20 %
proba = np.where(frais > np.median(frais), 0.8, 0.2)
ecoles = pd.DataFrame({"frais": frais, "tablette": np.random.binomial(1, proba)})

# La verite, que nous seuls connaissons : la tablette fait PERDRE 50 points
ecoles["y0"] = (200 + 0.7 * frais + np.random.normal(0, 80, n)).round(1)
ecoles["y1"] = (ecoles["y0"] - 50).round(1)

# Chaque ecole ne vit qu'un seul des deux mondes
ecoles["score"] = np.where(ecoles["tablette"] == 1, ecoles["y1"], ecoles["y0"])
ecoles.head(3)

### Ce que voit l'observateur

Il ne voit ni `y0` ni `y1` : il ne voit que `score`, le résultat qui s'est
réellement produit. Comparons les deux groupes, comme le ferait n'importe
quel tableau de bord.

In [ ]:
ecoles.groupby("tablette")["score"].mean().round(1)

In [ ]:
plt.figure(figsize=(7, 4))
for valeur, couleur, nom in [(0, "steelblue", "sans tablette"), (1, "salmon", "avec tablette")]:
    part = ecoles.query("tablette == @valeur")
    plt.scatter(part["frais"], part["score"], s=10, alpha=0.4, color=couleur, label=nom)

plt.xlabel("frais de scolarite (EUR)")
plt.ylabel("score au test")
plt.title("Les tablettes vont aux ecoles cheres")
plt.legend()
plt.show()

## L'erreur silencieuse

Le nuage penche vers le haut, les moyennes le confirment. Concluons donc que
la tablette apporte environ **+134 points**.

Sauf que nous avons écrit la vérité nous-mêmes, deux cellules plus haut.
Regardons-la.

In [ ]:
naif = (ecoles.query("tablette == 1")["score"].mean()
        - ecoles.query("tablette == 0")["score"].mean())

# La verite, accessible seulement parce que c'est une simulation
vrai = (ecoles["y1"] - ecoles["y0"]).mean()

print("comparaison naive :", round(naif, 1), "points")
print("effet reel        :", round(vrai, 1), "points")

**+134 contre −50.** Le code a tourné sans un avertissement, et il s'est
trompé **de signe**. Une décision prise sur ce chiffre distribuerait des
tablettes pour dégrader les résultats.

C'est la forme d'erreur la plus dangereuse du métier : celle qui ne prévient
pas. Le reste de la séance sert à la nommer, à la mesurer, et à s'en
débarrasser.

## Le vocabulaire minimal

Trois notations, et tout le bloc tient dedans.

- $T_i$ : le **traitement** reçu par l'individu $i$. $T_i = 1$ s'il l'a reçu,
  $0$ sinon. Ici : recevoir une tablette, recevoir un email.
- $Y_i$ : le **résultat** observé pour $i$ — ce qu'on cherche à améliorer.
  Le score au test, la dépense du client.
- $Y_{0i}$ et $Y_{1i}$ : les deux **résultats potentiels**. Ce que $i$ aurait
  obtenu *sans* traitement, et *avec*.

Le mot important est **potentiels** : ils existent tous les deux, mais un seul
se réalise. Celui qui ne s'est pas produit s'appelle le **contrefactuel**.

> **Le problème fondamental de l'inférence causale :** pour un individu donné,
> on n'observe **jamais** les deux. Deux chemins divergent dans un bois, on
> n'en emprunte qu'un, et on ne saura jamais ce qu'il y avait sur l'autre.

Avec ces deux quantités, on peut définir ce qu'on cherche :

$$ATE = E[Y_1 - Y_0] \qquad\qquad ATT = E[Y_1 - Y_0 \mid T = 1]$$

L'**ATE** est l'effet moyen sur toute la population. L'**ATT** est l'effet
moyen sur les seuls individus traités. Ils diffèrent dès que le traitement
n'agit pas pareil sur tout le monde.

### Quatre écoles, et des pouvoirs divins

Imaginons un instant qu'on puisse voir les deux colonnes. `TE` est l'effet
individuel, $Y_1 - Y_0$.

In [ ]:
divin = pd.DataFrame({
    "ecole": [1, 2, 3, 4],
    "y0": [500, 600, 800, 700],
    "y1": [450, 600, 600, 750],
    "T":  [0, 0, 1, 1],
})
divin["TE"] = divin["y1"] - divin["y0"]
print("ATE :", divin["TE"].mean())
print("ATT :", divin.query("T == 1")["TE"].mean())
divin

$ATE = -50$, $ATT = -75$. Les tablettes dégradent les résultats, et un peu
plus fort dans les écoles qui en ont reçu.

Voici maintenant le **même tableau tel qu'on l'observe vraiment** : chaque
école ne livre qu'une seule de ses deux colonnes.

In [ ]:
reel = divin.copy()
reel["y0"] = np.where(reel["T"] == 1, np.nan, reel["y0"])
reel["y1"] = np.where(reel["T"] == 0, np.nan, reel["y1"])
reel["TE"] = np.nan
reel["Y"] = divin["y0"].where(divin["T"] == 0, divin["y1"])
reel

La colonne `TE` est vide, et le restera toujours. Alors on se rabat sur la
comparaison des moyennes observées : $(600 + 750)/2 - (500 + 600)/2 = +125$.

Le vrai ATE vaut $-50$. Même erreur, même inversion de signe, sur quatre
lignes qu'on peut vérifier à la main.

## Le biais de sélection, et sa formule

Pourquoi la comparaison naïve échoue-t-elle ? Écrivons-la et transformons-la.
On part de l'association, $E[Y \mid T=1] - E[Y \mid T=0]$, on ajoute et on
retranche le même terme $E[Y_0 \mid T=1]$ — le contrefactuel des traités — et
on réorganise :

$$
E[Y \mid T=1]-E[Y \mid T=0]
=
\underbrace{E[Y_1-Y_0 \mid T=1]}_{\text{ATT — ce qu'on veut}}
+
\underbrace{E[Y_0 \mid T=1]-E[Y_0 \mid T=0]}_{\text{biais de sélection}}
$$

**Cette ligne contient tout le bloc.** Ce que mesure une comparaison de
moyennes, c'est l'effet causal **plus** un terme de biais.

Et le biais a un sens très concret : c'est l'écart qui séparerait les deux
groupes **si personne n'avait été traité**. Les écoles riches auraient de
meilleurs scores sans tablette. Ce terme n'est pas nul, donc la comparaison
naïve n'est pas l'effet causal.

Cet écart-là, lui, est mesurable — sur les caractéristiques d'avant.

In [ ]:
ecoles.groupby("tablette")["frais"].mean().round(0)

849 € contre 1 120 €. Les deux groupes **ne sont pas comparables**, et ce
n'est pas la tablette qui les distingue.

Voici ce que la comparaison de moyennes voit — et ce qu'elle ne voit pas.

In [ ]:
moyennes = ecoles.groupby("tablette")["score"].mean()
moyennes.index = ["sans tablette", "avec tablette"]

plt.figure(figsize=(7, 4))
moyennes.plot(kind="barh", color=["steelblue", "salmon"])
plt.xlabel("score moyen au test")
plt.title("Tout ce que voit une comparaison de moyennes")
plt.show()

### Le monde que seuls les dieux voient

Reprenons vingt-cinq écoles et affichons leurs **deux** résultats potentiels.
Le trait gris est l'effet individuel de la tablette : toujours vers le bas,
toujours de 50 points.

In [ ]:
echant = ecoles.sample(25, random_state=1)

plt.figure(figsize=(7, 4))
plt.vlines(echant["frais"], echant["y1"], echant["y0"], color="lightgrey")
plt.scatter(echant["frais"], echant["y0"], s=16, color="steelblue", label="Y0 — sans tablette")
plt.scatter(echant["frais"], echant["y1"], s=16, color="salmon", label="Y1 — avec tablette")
plt.xlabel("frais de scolarite (EUR)")
plt.ylabel("score au test")
plt.title("Les deux resultats potentiels de chaque ecole")
plt.legend()
plt.show()

Les traits sont courts et tous orientés vers le bas : voilà l'effet causal,
$-50$ points. La pente du nuage, elle, est bien plus forte — et elle ne doit
rien aux tablettes. C'est cette pente qui pollue la comparaison naïve.

## La solution : tirer au sort

La formule dit exactement ce qu'il faut obtenir : faire disparaître le terme
de biais, c'est-à-dire obtenir

$$E[Y_0 \mid T=1] = E[Y_0 \mid T=0]$$

— les deux groupes seraient identiques **si aucun n'avait été traité**.

Il y a un moyen simple et brutal de forcer cette égalité : **attribuer le
traitement au hasard**. Le tirage au sort ne regarde ni la richesse de
l'école, ni l'historique du client, ni rien d'autre. Formellement :

$$(Y_0, Y_1) \perp\!\!\!\perp T$$

> ⚠️ Cela ne signifie **pas** que le traitement est sans effet. Ce sont les
> résultats *potentiels* qui sont indépendants du tirage : le hasard n'a pas
> choisi les bons élèves. Le traitement, lui, agit bel et bien — c'est
> précisément ce qu'on veut mesurer.

Le biais disparaît, et la différence de moyennes **devient** l'effet causal :

$$E[Y \mid T=1]-E[Y \mid T=0]=E[Y_1-Y_0]=ATE$$

Vérifions-le sur nos écoles : redistribuons les tablettes à pile ou face.

In [ ]:
tirees = ecoles.sample(frac=0.5, random_state=42).index
ecoles["tab_alea"] = ecoles.index.isin(tirees).astype(int)
ecoles["score_alea"] = np.where(ecoles["tab_alea"] == 1, ecoles["y1"], ecoles["y0"])

naif_alea = (ecoles.query("tab_alea == 1")["score_alea"].mean()
             - ecoles.query("tab_alea == 0")["score_alea"].mean())

print("comparaison naive, monde observe   :", round(naif, 1))
print("comparaison naive, monde randomise :", round(naif_alea, 1))
print("effet reel                         :", round(vrai, 1))

**−53 contre −50.** Le même calcul, sur les mêmes écoles, avec les mêmes
résultats potentiels — mais un mécanisme d'attribution différent. Rien
d'autre n'a changé.

Les trois points ne coïncident pas exactement : il reste 3 points d'écart. Ce
n'est plus du biais, c'est du **hasard d'échantillonnage**. La randomisation
garantit l'égalité *en espérance*, pas sur un tirage particulier. Mesurer ce
qui reste de hasard, c'est le travail de la séance 3.2 — et nous le
réutiliserons tel quel.

In [ ]:
plt.figure(figsize=(7, 4))
for valeur, couleur, nom in [(0, "steelblue", "sans tablette"), (1, "salmon", "avec tablette")]:
    part = ecoles.query("tab_alea == @valeur")
    plt.scatter(part["frais"], part["score_alea"], s=10, alpha=0.4, color=couleur, label=nom)

plt.xlabel("frais de scolarite (EUR)")
plt.ylabel("score au test")
plt.title("Apres tirage au sort : les couleurs se melangent")
plt.legend()
plt.show()

Comparez cette figure à la première. Les deux couleurs ne se séparent plus
sur l'axe des frais : le traitement est réparti sur tout le spectre. C'est
l'image de « les groupes sont comparables ».

## Un vrai essai randomisé

2020, les universités basculent en ligne. Faut-il y rester ? Comparer les
étudiants des cours en ligne à ceux du présentiel ne vaut rien : le distanciel
attire les plus autonomes, qui réussiraient de toute façon.

Des économistes ont donc **tiré au sort le format du cours**, puis fait passer
le même examen à tout le monde. 323 étudiants.

In [ ]:
classe = pd.read_csv(BASE + "online_classroom.csv")

classe["format"] = np.select(
    [classe["format_ol"] == 1, classe["format_blended"] == 1],
    ["en_ligne", "mixte"],
    default="presentiel")

classe.groupby("format")["falsexam"].mean().round(2)

78,5 en présentiel contre 73,6 en ligne : **l'ATE du distanciel est de −4,9
points**. Et c'est tout — pas de correction, pas de modèle. Le tirage au sort
a fait le travail.

Mais un bon analyste **vérifie** que la randomisation a fonctionné. Le test
est simple : les variables mesurées **avant** le traitement doivent être
quasiment identiques d'un groupe à l'autre.

In [ ]:
classe.groupby("format")[["gender", "black", "white"]].mean().round(3)

`gender` et `white` sont très proches. `black` bouge davantage : 3 % contre
7 %. Sur 323 étudiants, le hasard laisse ce genre d'écart — c'est normal, et
même attendu. Des groupes **parfaitement** identiques seraient suspects.

Ce tableau porte un nom : le **test d'équilibre**. C'est la version observable
de $E[Y_0 \mid T=1] = E[Y_0 \mid T=0]$ : on ne verra jamais les $Y_0$ des
traités, mais on peut regarder leurs caractéristiques d'avant. Vous le
referez sur les 64 000 clients de l'étude de cas.

## Les données de l'étude de cas

Passons à la vraie affaire : 64 000 clients d'un site de vente en ligne,
répartis **au hasard** en trois groupes — aucun email, un email produits
hommes, un email produits femmes. Puis deux semaines d'observation.

In [ ]:
data = pd.read_csv(BASE + "hillstrom.csv")

print(data.shape)
data[["recency", "history", "segment", "visit", "conversion", "spend"]].head(3)

### Une erreur bruyante, pour changer

Le réflexe naturel, c'est de demander la moyenne de tout par groupe.

In [ ]:
data.groupby("segment").mean()

Ne lisez pas la traceback. **Seule la dernière ligne compte.** Selon la
version de pandas installée, elle ressemble à l'une de ces deux formes :

```
TypeError: agg function failed [how->mean,dtype->object]
TypeError: dtype 'str' does not support operation 'mean'
```

Le libellé change, le sens est le même : on a demandé une **moyenne sur du
texte**. `zip_code` et `channel` contiennent « Rural », « Phone »… La moyenne
de « Rural » n'existe pas.

La correction consiste à dire quelles colonnes on veut, ce qui est de toute
façon une meilleure habitude : sur une tablette, un tableau de douze colonnes
est illisible.

In [ ]:
data.groupby("segment")[["visit", "conversion", "spend"]].mean().round(4)

### Laquelle des deux erreurs est la plus dangereuse ?

Nous en avons vu deux aujourd'hui :

| L'erreur | Ce qui se passe | Ce que ça coûte |
|---|---|---|
| `groupby().mean()` sur du texte | traceback rouge, la cellule s'arrête | trois minutes |
| comparer deux groupes non comparables | **rien**, un chiffre s'affiche | un budget |

La deuxième ne prévient jamais. C'est pour elle qu'existe tout ce bloc.

Et ce tableau, lui, est **honnête** : les trois groupes ont été tirés au sort,
donc les écarts qu'il montre sont des effets causals. 1,42 € contre 0,65 € de
dépense moyenne. Nous les mesurerons proprement en étude de cas.

## Quand on ne peut pas tirer au sort

L'essai randomisé est la référence — les autorités de santé l'exigent avant
d'autoriser un médicament. Si on pouvait, on ne ferait que ça.

Mais souvent on ne peut pas : c'est trop cher, contraire à l'éthique, ou hors
de notre contrôle. On ne va pas tirer au sort les femmes enceintes qui
fumeront, ni les pays qui augmenteront leur salaire minimum.

Ce qui compte alors, c'est le **mécanisme d'attribution** : *qu'est-ce qui a
décidé qui recevait le traitement ?* Dans un essai randomisé, la réponse est
« le hasard », et tout devient simple. Ailleurs, il faut l'argumenter avec ce
qu'on sait du monde — les données seules ne le diront jamais.

> 💡 Même quand l'expérience est impossible, demandez-vous **quelle serait
> l'expérience parfaite**. La réponse éclaire presque toujours la méthode de
> repli.

## Et l'A/B testing dans tout ça ?

Nous n'avons pas prononcé le mot. « Essai contrôlé randomisé » vient du monde
académique ; « A/B test » vient de l'industrie. **C'est la même idée.**

On propose deux versions d'un produit — deux objets d'email, deux pages
d'accueil, deux prix — à une audience **tirée au sort**, et on compare un
indicateur choisi à l'avance.

| | Essai randomisé | A/B test |
|---|---|---|
| Le principe | tirage au sort | tirage au sort |
| La durée | mois, années | jours, semaines |
| L'intervention | souvent lourde | légère, réversible |
| Le terrain | médecine, économie | marketing, produit numérique |

L'A/B test est donc un **sous-ensemble** des essais randomisés : plus court,
plus léger, plus fréquent. Mais l'inférence causale y est exactement la même,
avec les mêmes pièges — dont celui de la slide du stagiaire.

### L'outil des deux prochaines heures

La séance 3.2 vous a donné `stats.ttest_ind` pour savoir si un écart dépasse
le hasard. Emballons-le une fois pour toutes, avec son intervalle de
confiance.

In [ ]:
def comparer(a, b):
    """Effet, intervalle a 95 % et p-value entre deux groupes."""
    effet = a.mean() - b.mean()
    es = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    p = stats.ttest_ind(a, b, equal_var=False).pvalue
    return pd.Series({"effet": effet, "bas_95": effet - 1.96 * es,
                      "haut_95": effet + 1.96 * es, "p_value": p})


comparer(data.query("segment == 'Mens E-Mail'")["spend"],
         data.query("segment == 'No E-Mail'")["spend"]).round(4)

L'email hommes fait dépenser **+0,77 € par client contacté**, et l'intervalle
de confiance — de 0,49 € à 1,05 € — ne contient pas zéro. C'est un effet
causal, mesuré, avec sa marge d'erreur.

Reste la question que la slide du stagiaire ne pose jamais : **est-ce que ça
vaut le coup ?** C'est le travail des deux prochaines heures.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| les résultats par groupe | `df.groupby("segment")[["visit", "spend"]].mean()` |
| un tableau d'équilibre | `df.groupby("segment").agg(passe=("history", "mean"))` |
| l'effet absolu | `moyenne_traites - moyenne_temoins` |
| l'effet relatif | `(moyenne_traites - moyenne_temoins) / moyenne_temoins` |
| l'incertitude sur cet effet | `comparer(groupe_a, groupe_b)` |
| la p-value seule | `stats.ttest_ind(a, b, equal_var=False).pvalue` |
| croiser deux variables | `df.pivot_table(values="spend", index="channel", columns="segment", aggfunc="mean")` |

## La formule à retenir

$$
E[Y \mid T=1]-E[Y \mid T=0]
=
\underbrace{E[Y_1-Y_0 \mid T=1]}_{\text{l'effet causal}}
+
\underbrace{E[Y_0 \mid T=1]-E[Y_0 \mid T=0]}_{\text{le biais de sélection}}
$$

Une comparaison de moyennes, c'est l'effet **plus** le biais. Le tirage au
sort est ce qui annule le second terme — et rien d'autre ne le fait
gratuitement.

## Les quatre réflexes de la séance

1. **Avant de conclure, faire le tableau d'équilibre.** Si les groupes
   diffèrent sur les variables mesurées *avant* le traitement, la
   randomisation n'a pas eu lieu — quoi qu'on vous ait dit. Ici : 1,95 $
   d'écart sur la vraie expérience, 340 $ dans le monde parallèle.

2. **Ne jamais conditionner sur une variable post-traitement.** « Le panier
   des clients qui ont cliqué », « la conversion des ouvreurs » : ces
   comparaisons sont fausses par construction, et elles tournent sans
   avertissement. C'est l'erreur n° 1 des analyses A/B en entreprise.

3. **Choisir honnêtement entre absolu et relatif.** +0,68 point et +119 %
   décrivent le même fait. Le relatif seul, quand le niveau de départ est
   minuscule, est vrai et trompeur à la fois.

4. **« Significatif » ne veut pas dire « rentable ».** Une p-value dit que
   l'effet n'est probablement pas nul. La décision demande sa taille, son
   intervalle, le coût de la campagne et ce qu'on n'a pas mesuré.